In [8]:
# # 사전설치 : pip install pillow
# import gradio as gr
# import tensorflow as tf
# import numpy as np
# from PIL import Image
# import requests  # URL에서 데이터를 가져오기 위해 HTTP 요청을 보내는 라이브러리
# from io import BytesIO #  메모리에서 바이트 데이터를 저장하고 읽는 기능
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image
import requests
from io import BytesIO
from langchain_ollama import ChatOllama

In [2]:
# TensorFlow MobileNetV2 모델 로드
model = tf.keras.applications.MobileNetV2(weights="imagenet")  # 사전 훈련된 ImageNet 가중치를 사용

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [9]:
OLLAMA_SERVER = "http://localhost:11434"  # 로컬 서버 주소
MODEL_NAME = "gemma2"  # 사용하려는 Ollama 모델 이름

In [3]:
def predict_image(image_url):
    try:
        # URL에서 이미지 가져오기
        response = requests.get(image_url)
        image = Image.open(BytesIO(response.content)).resize((224, 224))  # BytesIO 사용하여 이미지 열기

        # 이미지를 배열로 변환
        image_array = tf.keras.preprocessing.image.img_to_array(image) # 이미지를 숫자 배열로 전환
        # Keras 모델은 여러 이미지를 한 번에 처리할 수 있도록 배치 차원을 요구 (처리이미지수, 높이, 너비, 채널)
        image_array = tf.expand_dims(image_array, axis=0)  # 모델이 한 번에 여러 이미지를 처리하도록 배열 앞에 "배치"라는 차원을 추가
        # MobileNetV2의 preprocess_input 함수는 주로 픽셀 값의 범위를 [0, 255]에서 [-1, 1]사이로 훈련 시 사용했던 정규화 범위로 조정
        image_array = tf.keras.applications.mobilenet_v2.preprocess_input(image_array)  # 이미지 픽셀 값을 모델이 학습할 때 사용했던 범위로 조정 스케일링

        # 예측 수행
        predictions = model.predict(image_array)  # 이미지를 분류하여 1000개 클래스에 대한 확률을 출력
        decoded_predictions = tf.keras.applications.mobilenet_v2.decode_predictions(predictions, top=3)[0]  # 상위 3개 예측 결과 반환, [0]: 배치 중 몇 번째 이미지인지

        # Gradio Label 컴포넌트에 맞게 결과 형식 변경
        # 예: {"pizza": 0.95, "burger": 0.03, "salad": 0.02} 형식으로 반환
        # _ : 클래스ID → 필요 없어서 _ (버리는 변수), label: 클래스명 → 딕셔너리 키로 사용, prob: 확률 → 딕셔너리 값으로 사용
        result = {label: float(prob) for (_, label, prob) in decoded_predictions}
        return result

    except Exception as e:
        return {"error": 1.0}  # 에러 발생 시 기본값 반환

In [10]:
# Ollama를 사용해 음식 설명 생성
def get_food_description_with_langchain(food_name):
    """
    LangChain ChatOllama를 사용하여 음식 설명 생성
    """
    try:
        chat = ChatOllama(base_url=OLLAMA_SERVER, model=MODEL_NAME)
        prompt = f"{food_name}에 대해 특징, 효능, 요리 레시피 설명해줘.설명은 한국어로 해줘."
        response = chat.invoke(prompt)
        return response.content
    except Exception as e:
        return f"Failed to retrieve description: {e}"

In [14]:
# 이미지 예측 함수
def predict_image_with_description(image_url):
    """
    이미지 URL을 받아 음식 예측과 Ollama 설명을 반환
    """
    try:
        # URL에서 이미지 가져오기
        response = requests.get(image_url)
        image = Image.open(BytesIO(response.content)).resize((224, 224))  # BytesIO 사용하여 이미지 열기

        # 이미지를 배열로 변환
        image_array = tf.keras.preprocessing.image.img_to_array(image)  # 이미지를 숫자 배열로 전환
        image_array = tf.expand_dims(image_array, axis=0)  # 모델이 한 번에 여러 이미지를 처리할 수 있게 "배치"라는 차원을 추가
        image_array = tf.keras.applications.mobilenet_v2.preprocess_input(image_array)  # 이미지 픽셀 값을 모델이 학습할 때 사용했던 범위로 조정 전처리

        # 예측 수행
        predictions = model.predict(image_array)
        decoded_predictions = tf.keras.applications.mobilenet_v2.decode_predictions(predictions, top=3)[0]  # 상위 3개 예측 결과 반환

        # 예측 결과 형식화
        result = {label: float(prob) for (_, label, prob) in decoded_predictions} # 예측 결과를 Gradio의 Label 컴포넌트가 요구하는 형식으로 변환

        # 가장 높은 확률의 예측값으로 Ollama 설명 생성
        top_food = decoded_predictions[0][1]  # 가장 확률이 높은 음식 이름
        description = get_food_description_with_langchain(top_food)

        return result, description  # 예측 결과와 Ollama 설명 반환

    except Exception as e:
        return {"error": 1.0}, f"Error: {e}"  # 에러 발생 시 기본값 반환

In [15]:
# Gradio 인터페이스 생성
iface = gr.Interface(
    #fn=predict_image,
    fn = predict_image_with_description,
    inputs=gr.Textbox(label="이미지 URL 입력"),
    outputs=[
        gr.Label(num_top_classes=3, label="예측 결과"),
        gr.Textbox(label="음식 설명", interactive=False)  # Ollama로 생성한 설명 출력
    ],
    title="음식 이미지 분류",
    description="이미지 URL을 입력하면 상위 3개의 예측 결과를 확률과 함께 표시합니다."
)

In [16]:
# 인터페이스 실행
iface.launch(server_name="0.0.0.0", debug=True)

# 예시 이미지 URL : https://health.chosun.com/site/data/img_dir/2024/04/19/2024041901914_0.jpg

* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


c:\human\AI_4\.venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
Keyboard interruption in main thread... closing server.


In [17]:
iface.close()

Closing server running on port: 7860


In [19]:
# 사전 설치 : pip install fpdf
import os
import gradio as gr
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser # LCEL과 함께 사용할 출력 파서
from fpdf import FPDF

In [20]:
# Ollama 설정 (Gemma2 모델 사용)
os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"  # Ollama 서버 주소
ollama_model = Ollama(model="gemma2")

C:\Users\human-17\AppData\Local\Temp\ipykernel_15736\2509825096.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  ollama_model = Ollama(model="gemma2")


In [21]:
# 다양한 템플릿 설정
TEMPLATES = {
    "취업": "다음 키워드와 예시를 바탕으로, 취업 지원을 위한 자기소개서를 작성하세요.",
    "대학원": "제공된 키워드를 사용하여, 대학원 지원을 위한 자기소개서를 초안 작성하세요.",
    "봉사활동": "주어진 키워드를 활용하여, 봉사활동 경험과 동기를 강조하는 자기소개서를 작성하세요."
}

In [22]:
# 언어 지원: 한국어, 영어, 일본어
LANGUAGES = {
    "한국어": "Please write the response in Korean.",
    "영어": "Please write the response in English.",
    "일본어": "Please write the response in Japanese."
}

In [23]:
# 자동 키워드 추천 함수
def recommend_keywords(purpose):
    if purpose == "취업":
        return "책임감, 팀워크, 문제 해결 능력"
    elif purpose == "대학원":
        return "연구 열정, 창의력, 학업 성취도"
    elif purpose == "봉사활동":
        return "사회적 책임감, 희생정신, 리더십"
    else:
        return ""

In [24]:
# 자기소개서 작성 함수
def generate_statement(purpose, language, keywords, example_sentence=None):
    if purpose not in TEMPLATES:
        return "지원 목적을 올바르게 선택해주세요."
    if language not in LANGUAGES:
        return "언어를 올바르게 선택해주세요."

    # 템플릿 생성
    template = TEMPLATES[purpose] + "\n\nKeywords: {keywords}\n" + LANGUAGES[language]
    if example_sentence:
        template += f"\n\nExample sentence: {example_sentence}"

    prompt = PromptTemplate(input_variables=["keywords"], template=template)

    # ======  LCEL 체인 사용 ======
    # 프롬프트, LLM, 출력 파서를 파이프(|)로 간단하게 연결합니다.
    # StrOutputParser는 LLM의 출력에서 문자열만 깔끔하게 추출해줍니다.
    chain = prompt | ollama_model | StrOutputParser()

    # .invoke()에 입력 변수를 딕셔너리 형태로 전달합니다.
    response = chain.invoke({"keywords": keywords})

    return response

In [25]:
# PDF 저장 함수
def save_to_pdf(statement, filename="personal_statement.pdf"):
    pdf = FPDF()
    pdf.add_page()  # pdf 문서에 새로운 페이지 추가
    pdf.add_font('MalgunGothic', '', r'C:\Windows\Fonts\malgun.ttf', uni=True)  # '맑은 고딕' 폰트 경로 설정
    pdf.set_font('MalgunGothic', size=12)  # 폰트 설정
    # Mac용 폰트 경로 설정
    # font_path = "/System/Library/Fonts/AppleSDGothicNeo.ttc"
    # pdf.add_font('AppleSDGothic', '', font_path, uni=True)
    # pdf.set_font('AppleSDGothic', size=12)

    pdf.multi_cell(0, 10, statement) # PDF 문서에 텍스트를 추가, 셀의너비(0), 셀의 높이(10)
    pdf.output(filename) # PDF 문서를 파일로 저장
    return f"PDF 저장 완료: {filename}"

In [26]:
# Gradio 인터페이스
def chatbot_interface(purpose, language, keywords, example_sentence=None, save_pdf=False):
    statement = generate_statement(purpose, language, keywords, example_sentence)
    if save_pdf:
        save_to_pdf(statement)
    return statement

In [33]:
with gr.Blocks() as demo:
    gr.Markdown("# 다목적 자기소개서 작성 도우미")
    gr.Markdown("키워드와 추천 문장을 활용하여 취업, 대학원, 봉사활동 자기소개서를 생성하고 PDF로 저장하세요!")

    # 입력 영역
    with gr.Row():
        purpose_input = gr.Dropdown(label="지원 목적", choices=["취업", "대학원", "봉사활동"], value="취업")
        language_input = gr.Dropdown(label="언어 선택", choices=["한국어", "영어", "일본어"], value="한국어")

    recommended_keywords = gr.Textbox(label="추천 키워드", interactive=False)
    recommend_btn = gr.Button("키워드 추천")
    recommend_btn.click(recommend_keywords, inputs=[purpose_input], outputs=[recommended_keywords])

    with gr.Row():
        keywords_input = gr.Textbox(label="사용자 키워드 입력", placeholder="예: 책임감, 팀워크, 문제 해결 능력")
        example_sentence_input = gr.Textbox(
            label="추천 문장 (선택 사항)",
            placeholder="예: '저는 도전을 두려워하지 않고 성공적으로 프로젝트를 완수했습니다.'"
        )

    save_pdf_toggle = gr.Checkbox(label="PDF로 저장", value=False)

    # 출력 영역
    output = gr.Textbox(label="작성된 자기소개서", lines=6)
    submit_btn = gr.Button("작성하기")
    submit_btn.click(
        fn=chatbot_interface,
        inputs=[purpose_input, language_input, keywords_input, example_sentence_input, save_pdf_toggle],
        outputs=[output]
    )

In [34]:
# 실행
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\human\AI_4\.venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [35]:
demo.close()

Closing server running on port: 7861
